# 🏆 Speech Emotion Recognition — Results & Comparative Analysis

## 🎯 Objectif
Analyser et comparer les performances des 3 modèles entraînés :
1. **CNN + BiLSTM** (features acoustiques manuelles)
2. **Wav2Vec2** (représentations auto-supervisées)
3. **HuBERT** (représentations auto-supervisées)

## 📊 Métriques
- Accuracy, Precision, Recall, F1-Score (global et par émotion)
- Matrices de confusion
- Courbes d’entraînement (Loss, Accuracy)
- Analyse des erreurs
- Comparaison statistique

In [ ]:
import os
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, precision_score, recall_score
)

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

# Charger les classes d'émotions
classes = np.load('label_classes.npy', allow_pickle=True)
print('Classes :', classes)

In [ ]:
# Charger les historiques d'entraînement
model_names = ['cnn_bilstm', 'wav2vec2', 'hubert']
display_names = {'cnn_bilstm': 'CNN+BiLSTM', 'wav2vec2': 'Wav2Vec2', 'hubert': 'HuBERT'}
model_colors = {'cnn_bilstm': '#2196F3', 'wav2vec2': '#FF9800', 'hubert': '#4CAF50'}

histories = {}
for name in model_names:
    path = f'histories/{name}_history.json'
    if os.path.exists(path):
        with open(path, 'r') as f:
            histories[name] = json.load(f)
        print(f'{display_names[name]:15s} | Epochs: {len(histories[name]["train_loss"])}')
    else:
        print(f'{display_names[name]:15s} | NOT FOUND')

# Charger les prédictions
predictions = {}
for name in model_names:
    path = f'predictions/{name}_preds.npz'
    if os.path.exists(path):
        data = np.load(path)
        predictions[name] = {'preds': data['preds'], 'labels': data['labels']}

print(f'\nMod\u00e8les charg\u00e9s : {len(predictions)}')

---
## 1. Courbes d'Entraînement

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 10))

for idx, name in enumerate(model_names):
    if name not in histories:
        continue
    h = histories[name]
    color = model_colors[name]
    
    # Loss
    axes[0, idx].plot(h['train_loss'], label='Train', color=color, linewidth=2)
    axes[0, idx].plot(h['val_loss'], label='Validation', color=color, linewidth=2, linestyle='--')
    axes[0, idx].set_title(f'{display_names[name]} - Loss')
    axes[0, idx].set_xlabel('Epoch')
    axes[0, idx].set_ylabel('Loss')
    axes[0, idx].legend()
    axes[0, idx].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1, idx].plot(h['train_acc'], label='Train', color=color, linewidth=2)
    axes[1, idx].plot(h['val_acc'], label='Validation', color=color, linewidth=2, linestyle='--')
    axes[1, idx].set_title(f'{display_names[name]} - Accuracy')
    axes[1, idx].set_xlabel('Epoch')
    axes[1, idx].set_ylabel('Accuracy')
    axes[1, idx].legend()
    axes[1, idx].grid(True, alpha=0.3)

plt.suptitle('Courbes d\'entra\u00eenement par mod\u00e8le', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Comparaison sur un seul graphique
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for name in model_names:
    if name not in histories:
        continue
    h = histories[name]
    color = model_colors[name]
    label = display_names[name]
    
    axes[0].plot(h['val_loss'], label=label, color=color, linewidth=2)
    axes[1].plot(h['val_acc'], label=label, color=color, linewidth=2)

axes[0].set_title('Validation Loss - Comparaison')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].set_title('Validation Accuracy - Comparaison')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 2. Métriques sur le Test Set

In [ ]:
# Tableau comparatif global
results = []

for name in model_names:
    if name not in predictions:
        continue
    preds = predictions[name]['preds']
    labels = predictions[name]['labels']
    
    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average='macro')
    f1_weighted = f1_score(labels, preds, average='weighted')
    prec = precision_score(labels, preds, average='macro')
    rec = recall_score(labels, preds, average='macro')
    
    results.append({
        'Mod\u00e8le': display_names[name],
        'Accuracy': f'{acc:.4f}',
        'F1 (macro)': f'{f1_macro:.4f}',
        'F1 (weighted)': f'{f1_weighted:.4f}',
        'Precision': f'{prec:.4f}',
        'Recall': f'{rec:.4f}'
    })

df_results = pd.DataFrame(results)
print('\n\u2550' * 80)
print('TABLEAU COMPARATIF - PERFORMANCES SUR LE TEST SET')
print('\u2550' * 80)
display(df_results.style.highlight_max(axis=0, subset=['Accuracy', 'F1 (macro)', 'F1 (weighted)'], color='lightgreen'))

In [ ]:
# Classification reports détaillés
for name in model_names:
    if name not in predictions:
        continue
    preds = predictions[name]['preds']
    labels = predictions[name]['labels']
    
    print(f'\n{"=" * 60}')
    print(f'CLASSIFICATION REPORT : {display_names[name]}')
    print(f'{"=" * 60}')
    print(classification_report(labels, preds, target_names=classes, digits=4))

---
## 3. Matrices de Confusion

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

for idx, name in enumerate(model_names):
    if name not in predictions:
        continue
    preds = predictions[name]['preds']
    labels = predictions[name]['labels']
    
    cm = confusion_matrix(labels, preds)
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=classes, yticklabels=classes,
                linewidths=0.5, linecolor='white', ax=axes[idx])
    
    acc = accuracy_score(labels, preds)
    axes[idx].set_title(f'{display_names[name]}\nAccuracy: {acc:.2%}', fontsize=13)
    axes[idx].set_xlabel('Pr\u00e9dit')
    axes[idx].set_ylabel('Vrai')
    axes[idx].tick_params(axis='x', rotation=45)

plt.suptitle('Matrices de confusion (valeurs brutes)', fontsize=15, y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
# Matrices de confusion normalisées (%)
fig, axes = plt.subplots(1, 3, figsize=(22, 6))

for idx, name in enumerate(model_names):
    if name not in predictions:
        continue
    preds = predictions[name]['preds']
    labels = predictions[name]['labels']
    
    cm = confusion_matrix(labels, preds, normalize='true') * 100
    
    sns.heatmap(cm, annot=True, fmt='.1f', cmap='YlOrRd', 
                xticklabels=classes, yticklabels=classes,
                linewidths=0.5, linecolor='white', ax=axes[idx],
                vmin=0, vmax=100)
    
    axes[idx].set_title(f'{display_names[name]} (normalis\u00e9)', fontsize=13)
    axes[idx].set_xlabel('Pr\u00e9dit')
    axes[idx].set_ylabel('Vrai')
    axes[idx].tick_params(axis='x', rotation=45)

plt.suptitle('Matrices de confusion normalis\u00e9es (%)', fontsize=15, y=1.03)
plt.tight_layout()
plt.show()

---
## 4. Analyse par Émotion

In [ ]:
# F1-Score par émotion pour chaque modèle
f1_per_emotion = {}

for name in model_names:
    if name not in predictions:
        continue
    preds = predictions[name]['preds']
    labels = predictions[name]['labels']
    f1_per_class = f1_score(labels, preds, average=None)
    f1_per_emotion[display_names[name]] = f1_per_class

df_f1 = pd.DataFrame(f1_per_emotion, index=classes)

# Barplot
ax = df_f1.plot(kind='bar', figsize=(14, 6), width=0.75,
                color=[model_colors[n] for n in model_names if n in predictions])
ax.set_title('F1-Score par \u00e9motion et par mod\u00e8le', fontsize=14)
ax.set_xlabel('\u00c9motion')
ax.set_ylabel('F1-Score')
ax.set_ylim(0, 1.05)
ax.legend(title='Mod\u00e8le')
ax.tick_params(axis='x', rotation=45)

# Ajouter les valeurs
for container in ax.containers:
    ax.bar_label(container, fmt='%.2f', fontsize=8)

plt.tight_layout()
plt.show()

print('\nF1-Score d\u00e9taill\u00e9 :')
display(df_f1.round(4))

In [ ]:
# Radar chart des performances par émotion
from matplotlib.patches import FancyBboxPatch

angles = np.linspace(0, 2 * np.pi, len(classes), endpoint=False).tolist()
angles += angles[:1]  # Fermer le polygone

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

for name in model_names:
    if name not in predictions:
        continue
    preds = predictions[name]['preds']
    labels = predictions[name]['labels']
    f1_values = f1_score(labels, preds, average=None).tolist()
    f1_values += f1_values[:1]
    
    ax.plot(angles, f1_values, 'o-', linewidth=2, label=display_names[name], color=model_colors[name])
    ax.fill(angles, f1_values, alpha=0.1, color=model_colors[name])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(classes, fontsize=11)
ax.set_ylim(0, 1)
ax.set_title('F1-Score par \u00e9motion (Radar Chart)', fontsize=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
ax.grid(True)

plt.tight_layout()
plt.show()

---
## 5. Analyse des Erreurs

In [ ]:
# Identifier les confusions les plus fréquentes pour chaque modèle
print('\u2550' * 70)
print('TOP 5 CONFUSIONS LES PLUS FR\u00c9QUENTES')
print('\u2550' * 70)

for name in model_names:
    if name not in predictions:
        continue
    preds = predictions[name]['preds']
    labels = predictions[name]['labels']
    
    cm = confusion_matrix(labels, preds)
    np.fill_diagonal(cm, 0)  # Ignorer les bonnes pr\u00e9dictions
    
    # Trouver les top confusions
    confusions = []
    for i in range(len(classes)):
        for j in range(len(classes)):
            if cm[i, j] > 0:
                confusions.append((classes[i], classes[j], cm[i, j]))
    
    confusions.sort(key=lambda x: x[2], reverse=True)
    
    print(f'\n--- {display_names[name]} ---')
    for true_label, pred_label, count in confusions[:5]:
        print(f'  {true_label:>10s} \u2192 {pred_label:<10s} : {count} erreurs')

In [ ]:
# Accord entre les modèles (tous prédisent la même chose)
if len(predictions) == 3:
    all_preds = np.stack([predictions[n]['preds'] for n in model_names])
    labels = predictions[model_names[0]]['labels']
    
    # Cas o\u00f9 tous les mod\u00e8les sont d'accord
    all_agree = np.all(all_preds == all_preds[0], axis=0)
    all_correct = np.all(all_preds == labels, axis=0)
    all_wrong = np.all(all_preds != labels, axis=0)
    
    n_total = len(labels)
    print(f'Total test samples: {n_total}')
    print(f'Tous les mod\u00e8les d\'accord: {all_agree.sum()} ({all_agree.sum()/n_total:.1%})')
    print(f'Tous corrects: {all_correct.sum()} ({all_correct.sum()/n_total:.1%})')
    print(f'Tous incorrects: {all_wrong.sum()} ({all_wrong.sum()/n_total:.1%})')
    
    # Ensemble voting
    from scipy import stats
    ensemble_preds = stats.mode(all_preds, axis=0)[0].flatten()
    ensemble_acc = accuracy_score(labels, ensemble_preds)
    ensemble_f1 = f1_score(labels, ensemble_preds, average='macro')
    print(f'\n\u2b50 Ensemble (Majority Voting):')
    print(f'  Accuracy: {ensemble_acc:.4f}')
    print(f'  F1 (macro): {ensemble_f1:.4f}')

---
## 6. Visualisation Comparative Finale

In [ ]:
# Barplot des métriques globales
metrics_data = []
for name in model_names:
    if name not in predictions:
        continue
    preds = predictions[name]['preds']
    labels = predictions[name]['labels']
    
    metrics_data.append({
        'Mod\u00e8le': display_names[name],
        'Accuracy': accuracy_score(labels, preds),
        'F1 (macro)': f1_score(labels, preds, average='macro'),
        'Precision': precision_score(labels, preds, average='macro'),
        'Recall': recall_score(labels, preds, average='macro'),
    })

df_metrics = pd.DataFrame(metrics_data)

# Ensemble
if len(predictions) == 3:
    df_metrics = pd.concat([df_metrics, pd.DataFrame([{
        'Mod\u00e8le': 'Ensemble',
        'Accuracy': ensemble_acc,
        'F1 (macro)': ensemble_f1,
        'Precision': precision_score(labels, ensemble_preds, average='macro'),
        'Recall': recall_score(labels, ensemble_preds, average='macro'),
    }])], ignore_index=True)

df_melted = df_metrics.melt(id_vars='Mod\u00e8le', var_name='M\u00e9trique', value_name='Score')

plt.figure(figsize=(14, 6))
ax = sns.barplot(data=df_melted, x='M\u00e9trique', y='Score', hue='Mod\u00e8le',
                 palette=['#2196F3', '#FF9800', '#4CAF50', '#9C27B0'][:len(df_metrics)])
ax.set_title('Comparaison globale des m\u00e9triques', fontsize=14)
ax.set_ylim(0, 1.05)

for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', fontsize=9)

plt.legend(title='Mod\u00e8le')
plt.tight_layout()
plt.show()

In [ ]:
# Learning Rate evolution
fig, ax = plt.subplots(figsize=(12, 4))

for name in model_names:
    if name not in histories:
        continue
    h = histories[name]
    ax.plot(h['lr'], label=display_names[name], color=model_colors[name], linewidth=2)

ax.set_title('\u00c9volution du Learning Rate')
ax.set_xlabel('Epoch')
ax.set_ylabel('Learning Rate')
ax.set_yscale('log')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 7. Tableau Récapitulatif

In [ ]:
# Tableau final avec toutes les informations
summary = []

model_info = {
    'cnn_bilstm': {'type': 'Hybride CNN+RNN', 'input': 'MFCC+Chroma+Spectral', 'pretrained': 'Non'},
    'wav2vec2': {'type': 'Transformer (SSL)', 'input': 'Waveform brute', 'pretrained': 'LibriSpeech 960h'},
    'hubert': {'type': 'Transformer (SSL)', 'input': 'Waveform brute', 'pretrained': 'LibriSpeech 960h'}
}

for name in model_names:
    if name not in predictions:
        continue
    preds = predictions[name]['preds']
    labels = predictions[name]['labels']
    h = histories.get(name, {})
    
    best_val_acc = max(h.get('val_acc', [0]))
    n_epochs = len(h.get('train_loss', []))
    
    summary.append({
        'Mod\u00e8le': display_names[name],
        'Type': model_info[name]['type'],
        'Input': model_info[name]['input'],
        'Pr\u00e9-entra\u00een\u00e9': model_info[name]['pretrained'],
        'Epochs': n_epochs,
        'Best Val Acc': f'{best_val_acc:.4f}',
        'Test Accuracy': f'{accuracy_score(labels, preds):.4f}',
        'Test F1 (macro)': f'{f1_score(labels, preds, average="macro"):.4f}',
    })

df_summary = pd.DataFrame(summary)
print('\n' + '\u2550' * 100)
print('TABLEAU R\u00c9CAPITULATIF')
print('\u2550' * 100)
display(df_summary)

---
## 8. Conclusions et Discussion

### Analyse des résultats

#### CNN + BiLSTM
- **Avantages** : Rapide à entraîner, faible empreinte VRAM, interprétable (features manuelles)
- **Limites** : Dépendant de la qualité des features extraites manuellement
- **Performance attendue** : ~55-65% accuracy sur RAVDESS (8 émotions)

#### Wav2Vec2
- **Avantages** : Représentations apprises automatiquement, transfert learning depuis LibriSpeech
- **Limites** : VRAM importante, temps d'entraînement plus long
- **Performance attendue** : ~65-75% accuracy

#### HuBERT
- **Avantages** : Souvent supérieur à Wav2Vec2 sur les tâches de classification audio
- **Limites** : Mêmes contraintes mémoire que Wav2Vec2
- **Performance attendue** : ~68-78% accuracy

### Confusions fréquentes dans RAVDESS
- **calm ↔ neutral** : Très similaires acoustiquement
- **happy ↔ surprise** : Partage de caractéristiques d'énergie \u00e9lev\u00e9e
- **fearful ↔ sad** : Tonalit\u00e9s similaires

### Pistes d’amélioration
1. **Data augmentation** plus agressive (SpecAugment, mixup)
2. **Fine-tuning progressif** : dégeler plus de couches transformer
3. **Ensemble learning** : combiner les 3 modèles
4. **Cross-corpus evaluation** : tester sur d'autres datasets (IEMOCAP, EMO-DB)
5. **Attention mechanisms** plus sophistiqués pour le CNN+BiLSTM

In [ ]:
# Sauvegarder le rapport final
df_summary.to_csv('final_results.csv', index=False)
df_f1.to_csv('f1_per_emotion.csv')

print('Fichiers de r\u00e9sultats sauvegard\u00e9s :')
print('  - final_results.csv')
print('  - f1_per_emotion.csv')
print('\n\u2705 Analyse termin\u00e9e.')